# Shared Preprocessing Notebook
## DataCo Supply Chain Dataset — Version 5

This notebook explores the dataset and verifies the shared implementation in `src/preprocessing.py` for fraud, late delivery, sales revenue, and product quantity prediction.

**Dataset path:** `data/DataCoSupplyChainDataset.csv` (relative to the repository root). The CSV is intentionally not committed to Git.

**Shared split:** 80/20, `random_state=42`, stratified for classification. Imputation, encoding, and scaling are fitted on training data only.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'DataCoSupplyChainDataset.csv'
sys.path.insert(0, str(ROOT / 'src'))
from preprocessing import load_and_prepare_base, get_task_splits, BASE_DROP_COLS, QUANTITY_LEAKAGE_COLS

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found at {DATA_PATH}. Place the CSV in data/.')
df = load_and_prepare_base(str(DATA_PATH))
print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
display(df.head())

In [ ]:
## Dataset exploration
print('Columns:', df.columns.tolist())
print('Duplicate rows:', df.duplicated().sum())
missing = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_percent': (df.isna().mean()*100).round(2)})
display(missing[missing.missing_count > 0])
display(df.describe(include='all').T)

## 1. Target creation and availability

| Task | Source column | Prepared target | Type |
|---|---|---|---|
| Fraud | `Order Status == 'SUSPECTED_FRAUD'` | `target_fraud` | Classification |
| Late delivery | `Late_delivery_risk` | `target_late_delivery` | Classification |
| Sales revenue | `Sales` | `target_sales` | Regression |
| Product quantity | `Order Item Quantity` | `target_quantity` | Regression |

`Order Status` is used only to derive the fraud label. It is excluded from the model features to prevent the model from simply reading the answer.

In [ ]:
target_columns = ['target_fraud', 'target_late_delivery', 'target_sales', 'target_quantity']
for target in target_columns:
    print(f'\n{target}: missing={df[target].isna().sum()}')
    display(df[target].value_counts(dropna=False).head(20))

## 2. Leakage prevention

The shared preprocessing function removes `Order Status`, `Delivery Status`, `Days for shipping (real)`, and `shipping date (DateOrders)` from model inputs. It creates the fraud label before dropping `Order Status`.

For product quantity, `Order Item Quantity` is the target and is not included as a feature. Sales/profit outcome columns are also excluded from that task.

In [ ]:
required_drops = ['Order Status', 'Delivery Status', 'Days for shipping (real)', 'shipping date (DateOrders)']
print('Required leakage columns included in BASE_DROP_COLS:')
for col in required_drops:
    print(f'{col}: {col in BASE_DROP_COLS}')
assert all(col in BASE_DROP_COLS for col in required_drops)
assert 'Order Item Quantity' in df.columns
assert 'Order Item Quantity' not in BASE_DROP_COLS  # handled as target-specific exclusion
print('Leakage configuration checks passed.')

## 3. Run the shared split function

This cell calls the same `get_task_splits()` function used by the model scripts. Classification tasks are stratified; all tasks use the same 80/20 split settings and training-only fitted preprocessing.

In [ ]:
task_specs = [
    ('fraud', 'classification'),
    ('late_delivery', 'classification'),
    ('sales', 'regression'),
    ('product_quantity', 'regression'),
]
verification = {}
for task, task_type in task_specs:
    X_train, X_test, y_train, y_test = get_task_splits(df, task=task, seed=42)
    total = len(y_train) + len(y_test)
    assert len(y_train) == int(total * 0.8)
    assert len(y_test) == total - len(y_train)
    assert X_train.shape[1] == X_test.shape[1]
    verification[task] = {'train_rows': len(y_train), 'test_rows': len(y_test), 'features': X_train.shape[1]}
    print(f'{task}: train={len(y_train):,} ({len(y_train)/total:.0%}), test={len(y_test):,} ({len(y_test)/total:.0%}), features={X_train.shape[1]}')
    if task_type == 'classification':
        print('  train class proportions:', pd.Series(y_train).value_counts(normalize=True).sort_index().round(4).to_dict())
        print('  test class proportions: ', pd.Series(y_test).value_counts(normalize=True).sort_index().round(4).to_dict())
print('\nAll shared split checks passed.')
display(pd.DataFrame(verification).T)

## 4. Team verification checklist

- [x] One dataset path: `data/DataCoSupplyChainDataset.csv`.
- [x] Fraud target derived from `Order Status == 'SUSPECTED_FRAUD'` before that column is dropped from features.
- [x] Product quantity is the fourth task, using `Order Item Quantity`.
- [x] Shared 80/20 split, `random_state=42`, and stratification for classification.
- [x] Imputation, encoding, and scaling fitted on training data only.
- [ ] Run this notebook locally with the team's actual CSV and retain the output for evidence.
